# Feature Engineering

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import load_and_clean
from feature_engineering import add_engineered_features, ALL_FEATURE_COLS

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

df = load_and_clean()
X  = add_engineered_features(df)
y  = df['Total Wait Time (min)']

print(f'Total features: {X.shape[1]}')
print(X.columns.tolist())

## 1. New Engineered Features Preview

In [ ]:
engineered = ['Nurse Load', 'Is Peak Hour', 'Resource Pressure',
              'Urgency x Time', 'Urgency x Weekend', 'Large Facility']

X[engineered].describe().round(3)

## 2. Correlation of All Features with Wait Time

In [ ]:
corr = X.corrwith(y).sort_values()

colors = ['#ef4444' if v < 0 else '#22c55e' for v in corr.values]
corr.plot(kind='barh', color=colors, figsize=(10, 7))
plt.axvline(0, color='white', linewidth=0.8)
plt.title('All Features — Correlation with Total Wait Time')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

print(corr.round(3))

## 3. Resource Pressure vs Wait Time

In [ ]:
plt.scatter(X['Resource Pressure'], y, alpha=0.2, color='steelblue', s=10)
plt.title('Resource Pressure vs Total Wait Time')
plt.xlabel('Resource Pressure Score')
plt.ylabel('Wait Time (min)')
plt.tight_layout()
plt.show()

## 4. Peak Hour Impact

In [ ]:
peak_avg    = y[X['Is Peak Hour'] == 1].mean()
nonpeak_avg = y[X['Is Peak Hour'] == 0].mean()

plt.bar(['Peak Hours\n(Afternoon/Evening)', 'Non-Peak Hours'],
        [peak_avg, nonpeak_avg],
        color=['#ef4444', '#22c55e'])
plt.title('Avg Wait Time: Peak vs Non-Peak Hours')
plt.ylabel('Minutes')
plt.tight_layout()
plt.show()

print(f'Peak hour avg wait:     {peak_avg:.1f} min')
print(f'Non-peak hour avg wait: {nonpeak_avg:.1f} min')
print(f'Difference:             {peak_avg - nonpeak_avg:.1f} min')

## 5. Urgency x Time Interaction

In [ ]:
temp = pd.DataFrame({
    'Urgency x Time': X['Urgency x Time'],
    'Wait Time': y
})

grouped = temp.groupby('Urgency x Time')['Wait Time'].mean()

grouped.plot(kind='bar', color='mediumpurple')
plt.title('Avg Wait Time by Urgency × Time of Day Interaction')
plt.xlabel('Urgency Level × Time of Day')
plt.ylabel('Avg Wait Time (min)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Feature Correlation Heatmap (All Features)

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.3, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix (All Features)')
plt.tight_layout()
plt.show()

## 7. Final Feature Summary

In [ ]:
summary = pd.DataFrame({
    'Feature': X.columns,
    'Corr with Wait Time': X.corrwith(y).values,
    'Mean': X.mean().values,
    'Std': X.std().values,
}).set_index('Feature').sort_values('Corr with Wait Time')

print(summary.round(3).to_string())